# PhishGuard AI — Dataset exploration

Exploratory analysis of the processed training data (`data/processed/emails.csv`), built by
`python -m scripts.prepare_data` from the **SpamAssassin Public Corpus** (legitimate) and the
**Nazario Phishing Corpus** (phishing, CC BY 4.0, © Jose Nazario).

> ⚠️ The corpora contain real phishing. This notebook only computes **aggregate statistics**:
> it never opens links and does not display message bodies.

Goals: understand class balance and sources, compare the two classes, see which words separate
them, and check how often the rule-based indicators fire in each class.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from config.settings import get_settings
from ml.dataset import load_processed
from ml.preprocess import extract_urls, normalize_text

settings = get_settings()
df = load_processed(settings.processed_data_dir / "emails.csv")
stats = json.loads((settings.processed_data_dir / "dataset_stats.json").read_text(encoding="utf-8"))
df["class"] = df["label"].map({0: "legitimate", 1: "phishing"})
pd.set_option("display.precision", 3)
print(f"{len(df):,} emails")

7,529 emails


## 1. Preparation summary

Raw messages read, and what the cleaning step removed.

In [2]:
pd.Series(
    {
        **{f"raw: {k}": v for k, v in stats["raw_by_source"].items()},
        "skipped (unreadable)": stats["skipped_unreadable"],
        "skipped (empty after parsing)": stats["skipped_empty"],
        "duplicates removed": stats["duplicates_removed"],
        "final total": stats["final_total"],
    },
    name="messages",
).to_frame()

,messages
raw: spamassassin_easy_ham,2500
raw: spamassassin_easy_ham_2,1400
raw: spamassassin_hard_ham,250
raw: nazario,4586
skipped (unreadable),0
skipped (empty after parsing),44
duplicates removed,1163
final total,7529


## 2. Class balance and sources

In [3]:
balance = df.groupby(["class", "source"]).size().rename("emails").to_frame()
balance["share of class"] = balance["emails"] / balance.groupby(level=0)["emails"].transform("sum")
balance

emails  share of class
class      source                                         
legitimate spamassassin_easy_ham      2464           0.602
           spamassassin_easy_ham_2    1387           0.339
           spamassassin_hard_ham       240           0.059
phishing   nazario                    3438           1.000

In [4]:
df["class"].value_counts(normalize=True).rename("share").to_frame()

,share
class,
legitimate,0.543
phishing,0.457


The classes are reasonably balanced (≈ 54 % / 46 %), so accuracy is not misleading on its
own, but we still report precision, recall and F1 for the phishing class.

## 3. Message length

In [5]:
df["subject_chars"] = df["subject"].str.len()
df["body_chars"] = df["body"].str.len()
df["body_words"] = df["body"].str.split().str.len()
df.groupby("class")[["subject_chars", "body_chars", "body_words"]].median().rename(
    index=lambda c: f"median — {c}"
)

,subject_chars,body_chars,body_words
class,,,
median — legitimate,34.0,880.0,139.0
median — phishing,40.0,1315.0,163.0


In [6]:
df.groupby("class")["body_words"].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

,count,mean,std,min,10%,50%,90%,max
class,,,,,,,,
legitimate,4091.0,257.0,481.0,0.0,29.0,139.0,514.0,8580.0
phishing,3438.0,187.0,167.0,0.0,35.0,163.0,348.0,4315.0


## 4. Links

In [7]:
df["n_urls"] = [len(extract_urls(f"{s} {b}")) for s, b in zip(df["subject"], df["body"])]
links = df.groupby("class")["n_urls"].agg(
    mean="mean", median="median", with_link=lambda s: (s > 0).mean()
)
links["with_link"] = (links["with_link"] * 100).round(1).astype(str) + " %"
links

,mean,median,with_link
class,,,
legitimate,3.260,1.0,64.1 %
phishing,2.653,2.0,90.5 %


Links are far more common in phishing (about 90 %) than in legitimate mail (about 64 %),
but most legitimate emails contain links too: "contains a link" alone would flag many
legitimate messages. The URL rules therefore look at the *structure* of each link instead.

## 5. Words that separate the classes

Document frequency of each normalised word (share of emails of the class containing it).
URLs, email addresses and numbers are already replaced by placeholder tokens.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(preprocessor=normalize_text, min_df=20, binary=True, stop_words="english")
X = vec.fit_transform(df["subject"] + " " + df["body"])
terms = vec.get_feature_names_out()
is_phish = (df["label"] == 1).to_numpy()
doc_freq = pd.DataFrame(
    {
        "phishing": X[is_phish].mean(axis=0).A1,
        "legitimate": X[~is_phish].mean(axis=0).A1,
    },
    index=terms,
)
doc_freq["difference"] = doc_freq["phishing"] - doc_freq["legitimate"]
print("More frequent in PHISHING")
display(doc_freq.sort_values("difference", ascending=False).head(15).round(3))
print("More frequent in LEGITIMATE")
display(doc_freq.sort_values("difference").head(15).round(3))

More frequent in PHISHING


,phishing,legitimate,difference
account,0.564,0.024,0.540
dear,0.469,0.014,0.455
thank,0.442,0.018,0.425
click,0.380,0.041,0.339
email,0.461,0.136,0.325
access,0.342,0.051,0.291
security,0.330,0.050,0.281
link,0.346,0.070,0.276
customer,0.291,0.016,0.274
urltoken,0.905,0.641,0.263


More frequent in LEGITIMATE


,phishing,legitimate,difference
emailtoken,0.098,0.495,-0.397
wrote,0.002,0.340,-0.338
just,0.068,0.330,-0.262
list,0.022,0.274,-0.253
group,0.019,0.224,-0.204
linux,0.001,0.202,-0.201
users,0.053,0.241,-0.188
don,0.070,0.247,-0.177
think,0.017,0.185,-0.168
subscription,0.007,0.171,-0.164


**Reading.** Phishing vocabulary is about accounts, banks, verification and security.
The legitimate side is dominated by early-2000s technical mailing lists (Linux, lists, replies):
this is a **topic bias** of SpamAssassin, documented in the evaluation report — it is one reason
why PhishGuard combines the model with explicit rules.

## 6. How often do the rule-based indicators fire?

The rules (`services/email_analyzer.py`) are run on a random sample of each class. Only
content and link indicators can fire here: headers and attachments are not kept in the
processed dataset.

In [9]:
from collections import Counter

from services.email_analyzer import analyze_email
from services.signals import signal_label

sample = df.groupby("class", group_keys=False).sample(n=800, random_state=42)
rows = []
for cls, part in sample.groupby("class"):
    counter = Counter()
    scores = []
    for subject, body in zip(part["subject"], part["body"]):
        analysis = analyze_email(body, subject=subject)
        counter.update({s.code for s in analysis.signals})
        scores.append(analysis.rule_score)
    for code, n in counter.items():
        rows.append({"indicator": signal_label(code), "class": cls, "share": n / len(part)})
    print(f"{cls:<11} mean rule score {pd.Series(scores).mean():.3f}")

prevalence = (
    pd.DataFrame(rows)
    .pivot_table(index="indicator", columns="class", values="share", fill_value=0)
    .sort_values("phishing", ascending=False)
)
(prevalence * 100).round(1).astype(str) + " %"

legitimate  mean rule score 0.031


phishing    mean rule score 0.421


class,legitimate,phishing
indicator,,
Suspicious link,5.1 %,67.5 %
Urgency,5.4 %,27.9 %
Credential request,0.0 %,26.5 %
Generic greeting,0.0 %,11.6 %
Suspension threat,0.1 %,9.6 %
Money / prize lure,0.9 %,2.2 %
Misleading link text,0.1 %,0.1 %


## 7. Conclusions

* The data is balanced enough; duplicates were removed before any split.
* Links are more frequent in phishing (≈ 90 % vs 64 %), but common in both: link *structure* is more informative than link presence.
* Word frequencies confirm a strong phishing vocabulary **and** a topic bias in the legitimate
  corpus (old technical mailing lists).
* Rule indicators are clearly more frequent in phishing, but no single rule is decisive —
  which is why they are combined (noisy-OR) and blended with the model probability.

See `reports/evaluation_report.md` and `reports/engine_evaluation.md` for the evaluation on the
held-out test set.